# Codelab: Process PDF Forms with EasyOCR and PyTorch

## Learning Objectives
- Create a Python virtual environment.
- Install EasyOCR and dependencies.
- Convert PDF pages to images.
- Extract text using EasyOCR.
- Parse form fields.
- Save extracted data to CSV.

## Best Practices
- Use a virtual environment.
- Test with high-resolution scans.
- Set `gpu=False` if CUDA is unavailable.
- Validate OCR results before parsing.

## Step 1. Create a Virtual Environment
### Explanation
A virtual environment isolates project dependencies.

## Install Poppler
Windows: [Download Here](https://github.com/oschwartz10612/poppler-windows/releases/download/v26.02.0-0/Release-26.02.0-0.zip)

macOS/Linux: `brew install poppler`

## Run these commands in your terminal
- python -m venv .venv
- Windows: .venv\Scripts\activate
- macOS/Linux: source .venv/bin/activate

## Step 2. Install Required Packages
### Explanation
Install the libraries required for OCR, PDF conversion, image processing, deep learning, and CSV export.

In [1]:
!pip install easyocr pdf2image pillow torch torchvision pandas


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Step 3. Import Libraries
### Explanation
Import the modules used throughout the notebook.

In [2]:
import os
import csv
from PIL import Image
from pdf2image import convert_from_path
import easyocr

## Step 4. Convert PDF to Images
### Explanation
Convert each PDF page into a PIL image for OCR.

In [3]:
def convert_pdf_to_images(pdf_path):
    print(f"Converting {pdf_path} to images...")
    return convert_from_path(pdf_path)

## Step 5. Perform OCR
### Explanation
Initialize EasyOCR and extract text plus bounding boxes.

In [4]:
def perform_ocr_on_image(image_path_or_buf):
    reader = easyocr.Reader(['en'], gpu=True)
    print("Running PyTorch-based EasyOCR extraction...")
    return reader.readtext(image_path_or_buf)

## Step 6. Parse Form Fields
### Explanation
Match recognized labels with nearby values.

In [5]:
def parse_form_fields(ocr_results):
    extracted_data={
        "Candidate Name":"Not Found","Address":"Not Found","Occupation":"Not Found",
        "Voter Identification Number":"Not Found","Local Government/Area Council":"Not Found",
        "Ward":"Not Found","Delimitation":"Not Found","Educational Qualifications":"Not Found",
        "Vice President Nominee":"Not Found",
        "Sponsoring Party":"Not Found"
    }
    text_lines=[res[1].strip() for res in ocr_results]
    for i,line in enumerate(text_lines):
        if "Voter Identification" in line or "VIN" in line:
            if i+1<len(text_lines): extracted_data["Voter Identification Number"]=text_lines[i+1]
        elif "Local Government" in line:
            if i+1<len(text_lines): extracted_data["Local Government/Area Council"]=text_lines[i+1]
        elif "Ward" in line:
            if i+1<len(text_lines): extracted_data["Ward"]=text_lines[i+1]
        elif "Delimitation" in line:
            if i+1<len(text_lines): extracted_data["Delimitation"]=text_lines[i+1]
        elif "sponsored by" in line.lower():
            if i+1<len(text_lines): extracted_data["Sponsoring Party"]=text_lines[i+1]
    return extracted_data

## Step 7. Save Results to CSV
### Explanation
Append extracted records to a CSV file.

In [6]:
def save_to_csv(data,csv_path="form_extractions.csv"):
    file_exists=os.path.isfile(csv_path)
    with open(csv_path,"a",newline="",encoding="utf-8") as f:
        writer=csv.DictWriter(f,fieldnames=data.keys())
        if not file_exists: writer.writeheader()
        writer.writerow(data)
    print(f"Successfully saved data to {csv_path}")

## Step 8. Process the Form
### Explanation
Combine PDF conversion, OCR, parsing, CSV export, and cleanup.

In [7]:
def process_form(pdf_path,csv_path="form_extractions.csv"):
    images=convert_pdf_to_images(pdf_path)
    if not images:
        print("No pages found."); return
    first_page=images[0]
    temp_img_path="temp_page1.png"
    first_page.save(temp_img_path,"PNG")
    ocr_results=perform_ocr_on_image(temp_img_path)
    form_data=parse_form_fields(ocr_results)
    save_to_csv(form_data,csv_path)
    if os.path.exists(temp_img_path):
        os.remove(temp_img_path)
    return form_data

## Step 9. Execute the Application
### Explanation
Run the pipeline on Form-EC13A.pdf.

In [9]:
if __name__ == "__main__":
    pdf_file_path = os.path.join("form", "Form-EC13A-1.pdf")
    csv_output_path = "form_extractions.csv"

    try:
        if not os.path.exists(pdf_file_path):
            raise FileNotFoundError(f"Could not find the file at: {os.path.abspath(pdf_file_path)}")

        data = process_form(pdf_file_path, csv_output_path)
        print("\n--- Extracted Form Data ---")
        for key, value in data.items():
            print(f"{key}: {value}")

    except Exception as e:
        print(f"An error occurred: {e}")

Converting form/Form-EC13A-1.pdf to images...
Running PyTorch-based EasyOCR extraction...
Successfully saved data to form_extractions.csv

--- Extracted Form Data ---
Candidate Name: Not Found
Address: Not Found
Occupation: Not Found
Voter Identification Number: b_
Local Government/Area Council: C.
Ward: Minna
Delimitation: 7992
Educational Qualifications: Not Found
Vice President Nominee: Not Found
Sponsoring Party: party


## Expected Output
A CSV file named `form_extractions.csv` containing the extracted fields and printed progress messages.